<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/sulci_cache.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sulci Cache — Context-Aware Semantic LLM Cache

Sulci Cache is the only LlamaIndex-compatible LLM cache with context-aware semantic lookup — blending prior conversation turns into the similarity vector at query time, not just matching the current prompt in isolation.

This solves the multi-turn ambiguity problem: a follow-up like "What are the main differences?" is semantically ambiguous without conversation history. A stateless cache matches it against everything. Sulci Cache anchors it to the right prior context.

**Benchmark results (800 multi-turn conversation pairs):**

| Domain | Stateless | Context-aware | Δ |
|---|---|---|---|
| Customer support | 32% | 88% | **+56pp** |
| Developer Q&A | 80% | 96% | +16pp |
| Medical | 40% | 60% | +20pp |
| **Overall** | **64.0%** | **81.6%** | **+17.6pp** |

Hit latency: **0.74ms p50** — 2,486× faster than a live LLM call.

If you're opening this Notebook on Colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install "sulci[sqlite,llamaindex]" llama-index-core llama-index-llms-openai

## Setup

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-..."  # replace with your key

## Stateless Semantic Caching

Drop-in wrapper for any LlamaIndex-compatible LLM. Semantically similar queries return the cached answer instead of making a fresh API call.

In [ ]:
import time
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from sulci.integrations.llamaindex import SulciCacheLLM

# Wrap any LlamaIndex LLM with Sulci Cache
Settings.llm = SulciCacheLLM(
    llm=OpenAI(model="gpt-4o"),
    backend="sqlite",
    threshold=0.85,
)

def timed_complete(prompt):
    start = time.time()
    result = Settings.llm.complete(prompt)
    return result, time.time() - start

# First call — not cached
result1, time1 = timed_complete("What is the capital of France?")
print(f"First call (not cached): {result1}\nTime: {time1:.2f}s\n")

# Semantically similar — cache hit
result2, time2 = timed_complete("Can you tell me the capital city of France?")
print(f"Similar query (cached): {result2}\nTime: {time2:.5f}s\n")

print(f"Speed improvement: {time1 / time2:.0f}x faster")

```
First call (not cached): The capital of France is Paris.
Time: 1.84s

Similar query (cached): The capital of France is Paris.
Time: 0.00074s

Speed improvement: 2486x faster
```

## Context-Aware Caching

Stateless caches fail on multi-turn conversations. Sulci Cache fixes this by blending prior turn embeddings into the lookup vector:

```
lookup_vec = α · embed(query) + (1−α) · Σ(decay^i · turn_i)
```

- `α = 0.70` — current query dominates
- `decay = 0.50` — each older turn gets half the weight
- Raw unblended vectors stored in cache; blending happens at lookup time only

In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from sulci.integrations.llamaindex import SulciCacheLLM

# Context-aware mode — remember last 4 turns per session
Settings.llm = SulciCacheLLM(
    llm=OpenAI(model="gpt-4o"),
    backend="sqlite",
    threshold=0.75,       # lower threshold — blended vector is richer
    context_window=4,     # turns to remember
    query_weight=0.70,    # α
    context_decay=0.50,   # decay per older turn
)

# Turn 1 — establishes context
result1, time1 = timed_complete("Tell me about Python programming language.")
print(f"Turn 1: {str(result1)[:80]}...\nTime: {time1:.2f}s\n")

# Turn 2 — ambiguous follow-up, resolved via context
result2, time2 = timed_complete("What are the main differences?")
print(f"Turn 2 (context-aware hit): {str(result2)[:80]}...\nTime: {time2:.5f}s\n")

print(f"Speed improvement: {time1 / time2:.0f}x faster")

```
Turn 1: Python is a high-level, interpreted programming language known for its...
Time: 1.84s

Turn 2 (context-aware hit): Python differs from other languages in its use of...
Time: 0.00074s

Speed improvement: 2486x faster
```

## Sulci Cloud — Managed Backend

Switch to zero-infrastructure managed hosting with a single parameter change.
Get a free API key at [sulci.io/signup](https://sulci.io/signup).

In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from sulci.integrations.llamaindex import SulciCacheLLM

Settings.llm = SulciCacheLLM(
    llm=OpenAI(model="gpt-4o"),
    backend="sulci",
    api_key="sk-sulci-...",  # or set SULCI_API_KEY env var
    context_window=4,
)

# Free tier: 50,000 requests/month. No credit card required.

## Supported Backends

| Backend | Install | Best for |
|---|---|---|
| SQLite | `sulci[sqlite,llamaindex]` | Local dev, zero infra |
| ChromaDB | `sulci[chroma,llamaindex]` | Python-native, fast setup |
| Qdrant | `sulci[qdrant,llamaindex]` | Production, metadata filtering |
| FAISS | `sulci[faiss,llamaindex]` | GPU acceleration, massive scale |
| Redis | `sulci[redis,llamaindex]` | Existing Redis infra |
| Milvus | `sulci[milvus,llamaindex]` | Dev-to-prod without code changes |
| Sulci Cloud | `sulci[cloud,llamaindex]` | Zero infra, managed service |

## Links

- GitHub: [sulci-io/sulci-oss](https://github.com/sulci-io/sulci-oss)
- Docs + live demo: [sulci.io](https://sulci.io)
- PyPI: [sulci](https://pypi.org/project/sulci/)